In [ ]:
#----------------------------#
# IDEA
#----------------------------#
# 1.) Load and Prepare Data
# 2.) Basic Analysis
# 3.) Sankey Plot Basic
# 4.) Average Embedding Plot
# 5.) Sankey Plot + Position (UNFINISHED, problems with transformation)
#
# Possible solutions for 5.) 
# 5.0 current horrible solution: use matplotlib transformation and tweaking variables to make the plot visible
# 5.1 Bit different solution using interpolation (thx Raphi)
# also https://stackoverflow.com/questions/7343697/how-to-implement-linear-interpolation
# 5.2  these guys have an approach for figures from matplotlib to plotly, maybe helpful
#https://stackoverflow.com/questions/52615425/matplotlib-to-plotly-offline
# 5.3 fix axis?
# 5.4 find other suiting transformation for PCA values int 0-1 space
#
# TO DO: 
# - check effect of transformations (e.g interpolation) on PCA plot
# 

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import Counter
import matplotlib.pyplot as plt
import import_ipynb
import math
pd.options.display.max_rows = 1000

In [ ]:
home_path = %pwd
data_path = "/home/pavel/Documents/Keusch_Job/Data/"
plot_path  = "/home/pavel/Documents/Keusch_Job/Tasks/homogeneity/"
subroutines_path = "/home/pavel/Documents/Keusch_Job/Subroutines/"

In [ ]:
#Subroutines
%cd  $subroutines_path
%run "selected_subreddits.ipynb"
%run "data_loader.ipynb"
%run "basic_plots.ipynb"
%cd  $home_path

In [ ]:
#----------------------------#
# 1.) Load and Prepare Data
#----------------------------#

In [ ]:
#https://stackoverflow.com/questions/44906754/unique-values-of-two-columns-for-pandas-dataframe?rq=1
S_3 = load_data(data_name="SUBM_2020_03.csv", data_path = data_path)
S_p_3 = S_3.groupby(["author","subreddit"]).size().reset_index(name='Freq')
S_3.head()
del S_3

In [ ]:
C_3 = load_data(data_name="COMM_2020_03.csv", data_path = data_path)
C_p_3 = C_3.groupby(["author","subreddit"]).size().reset_index(name='Freq')
del C_3

In [ ]:
S_p_3

In [ ]:
id_corona = S_p_3.loc[S_p_3["subreddit"]=="COVID19",].copy()

In [ ]:
#print("Ok, so maybe a few bot cases, but certainly not many")
#print("WARNING: Does not mean that bots don't influence ")
print("TOP USERS;")
id_corona.sort_values("Freq",ascending=False)

In [ ]:
TM_SS = id_corona.merge(S_p_3.loc[S_p_3["subreddit"]!="COVID19",], how="inner", 
                left_on ="author", right_on ="author")

In [ ]:
TM_SS.head()

In [ ]:
TM_SC = id_corona.merge(C_p_3.loc[C_p_3["subreddit"]!="COVID19",], how="inner", 
                left_on ="author", right_on ="author")

In [ ]:
TM_SC.head()

In [ ]:
TM_Overall = TM_SS[["author","subreddit_y"]].copy()
TM_Overall = TM_Overall.append(TM_SC[["author","subreddit_y"]], ignore_index = True) 
# Next line is necessary to get two rows that are the same e.g Coronavirus greyco31
# however one is from the comments, one from the submissions.
TM_Overall = TM_Overall.groupby(["subreddit_y","author"]).size().reset_index(name ="Freq").sort_values("Freq", ascending=False)
TM_Overall.head()

In [ ]:
print(len(TM_SS))
print(len(TM_SC))
print(len(TM_Overall))

In [ ]:
#----------------------------#
# 2.) Basic Analysis
#----------------------------#

In [ ]:
print("Percentage accounts from COVID19 that have posted submission in other subreddits:")
print(len(id_corona.loc[id_corona["author"].isin(TM_SS["author"].tolist()),]) / len(id_corona))
print("Percentage accounts that COVID19 have posted comments in other subreddits:")
print(len(id_corona.loc[id_corona["author"].isin(TM_SC["author"].tolist()),]) / len(id_corona))
print("Percentage accounts that COVID19 have posted comments or submissions in other subreddits:")
print(len(id_corona.loc[id_corona["author"].isin(TM_Overall["author"].tolist()),]) / len(id_corona))

In [ ]:
#
print("% Accounts in COVID19 that are marked bots elsewhere")
print(len(id_corona.loc[(id_corona["author"].isin(TM_SS["author"].loc[TM_SS["Freq_y"]>100,].tolist())),]) /  len(id_corona))
print(len(id_corona.loc[(id_corona["author"].isin(TM_SC["author"].loc[TM_SC["Freq_y"]>100,].tolist())),]) /  len(id_corona))
print("This means very few of the accounts were bots")
print("WARNING: This does not mean that they didn't have influence")
print("WARNING: Also: only one possible bot definition")
#

#len(id_corona.loc[ (id_corona["author"].isin(TM_SC["author"].tolist()))
#              & (id_corona["Freq"]< 200),]) / len(id_corona.loc[ (id_corona["author"].isin(TM_SC["author"].tolist())),])

In [ ]:
#Group By
use_flow_ss = TM_SS.groupby("subreddit_y").size().reset_index(name="Freq").sort_values("Freq", ascending=False)
use_flow_ss

In [ ]:
print(use_flow_ss.describe())

In [ ]:
#Group By
use_flow_sc = TM_SC.groupby("subreddit_y").size().reset_index(name="Freq").sort_values("Freq", ascending=False)
use_flow_sc

In [ ]:
print(use_flow_sc.describe())

In [ ]:
#Group By
use_flow_overall = TM_Overall.groupby("subreddit_y").size().reset_index(name="Freq").sort_values("Freq", ascending=False)
use_flow_overall

In [ ]:
print(use_flow_overall.describe())

In [ ]:
simple_frequency(dat=use_flow_ss, path = plot_path, file_name="use_flow_ss", 
                            main_title="Frequency User Flows Submissions", 
                            main_color ='black',freq_var='Freq',
                             xlabel = "Frequency of User Flows to COVID19 by Submission", ylabel ='Frequency',
                             x_axis_vector=False, 
                             bins = 50,histtype="step", dpi = 600)

In [ ]:
simple_frequency(dat=use_flow_sc, path = plot_path, file_name="use_flow_sc", 
                            main_title="Frequency User Flows", 
                            main_color ='black',freq_var='Freq',
                             xlabel = "Frequency of User Flows to COVID19 by Comments", ylabel ='Frequency',
                             x_axis_vector=False, 
                             bins = 50,histtype="step", dpi = 600)

In [ ]:
simple_frequency(dat=use_flow_overall, path = plot_path, file_name="use_flow_overall", 
                            main_title="Frequency User Flows Overall", 
                            main_color ='black',freq_var='Freq',
                             xlabel = "Frequency of User Flows to COVID19 by Comments or Submissions", ylabel ='Frequency',
                             x_axis_vector=False, 
                             bins = 50,histtype="step", dpi = 600)

In [ ]:
#CONCLUSION: 2040 people that have posted on march in COVID19 have also posted in Coronavirus
# So most of the co-occurrences are with, politics, news and other Corona channels
# So 201 that have posted on COVID19 have commented or posted on science in march
#
# Further expansion
# 1.) Include PCA Coordinates (see below)
# 2.) have a "home subreddit". Flow is then the amount of users whos home is x that go to y
# 3.) weight user flow according to posts
# 4.) define flow from x to only if a user has poster in x BEFORE y
# 5.) VLAD: include semantic difference into homogeneity/ flow measure
#
# 6.) Continue with actual homogeneity thesis: Does semantic diversity cause boundaries
# However for hafendahl index you need to define flow measure

In [ ]:
#----------------------------#
# 3.) Sankey Plot Basic
#----------------------------#
# everything you need :)
# https://plotly.com/python/sankey-diagram/

In [ ]:
import plotly.graph_objects as go

def covid_sankey(flow_dat):
    label_list = flow_dat["subreddit_y"].tolist()
    label_list.insert(0,"COVID19")
    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 50,
          thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = label_list,
          color = "blue"
        ),
        link = dict(
          source = list(range(1,len(label_list))), # indices correspond to labels, eg A1, A2, A2, B1, ...
          target = [0] * len(label_list),
          value = flow_dat["Freq"].tolist()
      ))])
    fig.update_layout(title_text="Basic Sankey Diagram", font_size=10)
    fig.show()

In [ ]:
covid_sankey(use_flow_ss.loc[use_flow_ss["Freq"]>200,])

In [ ]:
covid_sankey(use_flow_ss.loc[use_flow_ss["Freq"] < 10,])

In [ ]:
covid_sankey(use_flow_ss)

In [ ]:
#----------------------------#
# 4.) Average Embedding Plot
#----------------------------#

In [ ]:
avg_embed_dict_3 = np.load(data_path + 'avg_emb_dict_3.npy',allow_pickle='TRUE').item()

In [ ]:
def dict_2_matrix(s_dict, key_list):
    #s_dict: some dictionary where the values are embedings ()
    #key_list: a list of keys
    #RETURNS: the np.array matrix where each value is a row.
    emb_matrix = np.append(s_dict[key_list[0]], s_dict[key_list[1]],axis=0)
    for i in range(2,len(key_list)):
        emb_matrix = np.append(emb_matrix, s_dict[key_list[i]],axis=0)
    return(emb_matrix)

In [ ]:
avg_embed_matrix_3 = dict_2_matrix(avg_embed_dict_3, list(avg_embed_dict_3.keys()))
avg_embed_matrix_3.shape

In [ ]:
# have to transpose because.... above the use the day dimensions as columns
avg_embed_matrix_3 = np.transpose(avg_embed_matrix_3)
avg_embed_matrix_3.shape

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
model = PCA(n_components=2).fit(avg_embed_matrix_3)

In [ ]:
%matplotlib inline
import seaborn as sns

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x'] + 0.0002, point['y'], str(point['val']), alpha = 0.5)

In [ ]:
def emb_plot(emb_matrix, plot_path = plot_path , file_name = "subreddits_avg_emb_PCA_labels",
             label_series = selected_subreddits["subreddit"], 
             legend = True, plot_labels= False, MDS = False):
    
    # PCA
    if MDS:
        category_vectors = MDS_embedding.fit_transform(emb_matrix.T)
    else:
        model = PCA(n_components=2).fit(emb_matrix)
        category_vectors = model.components_.T
    #    # transform df
    category_vector_frame=pd.DataFrame(category_vectors, index = label_series.tolist(), #index=subr_list, 
                                   columns=['col1', 'col2']).reset_index()
    #
    #Plot 
    plt.figure(figsize=(10, 10))
    if legend:
        sns.scatterplot(data=category_vector_frame, x='col1', y='col2',hue='index');
        plt.legend(loc='upper right')
    else:
        sns.scatterplot(data=category_vector_frame, x='col1', y='col2');
    if plot_labels:
        # need to do it because https://stackoverflow.com/questions/40339886/pandas-concat-generates-nan-values
        label_series_2 = label_series.copy()
        label_series_2.reset_index(drop=True, inplace=True)
        label_point(category_vector_frame.col1, category_vector_frame.col2,label_series_2, plt.gca())
    plt.savefig(fname=plot_path + file_name)

In [ ]:
subr_df_3 = pd.DataFrame(list(avg_embed_dict_3.keys()), columns = ["subreddit"])

In [ ]:
emb_plot(avg_embed_matrix_3,  plot_path = plot_path , file_name = "subreddits_avg_emb_PCA_labels_03",
        label_series = subr_df_3["subreddit"], legend = False, plot_labels= True)

In [ ]:
#----------------------------#
# 5.) Sankey Plot + Position (UNFINISHED)
#----------------------------#

In [ ]:
pca_df_3 = pd.DataFrame(model.components_.T, index = subr_df_3["subreddit"].tolist(), #index=subr_list, 
                                   columns=['col1', 'col2']).reset_index()
pca_df_3.columns = ["subreddit","PCA_x","PCA_y"]

In [ ]:
use_flow_ss = use_flow_ss.merge(pca_df_3, how = "inner", left_on = "subreddit_y",right_on="subreddit")
del use_flow_ss["subreddit"]

In [ ]:
use_flow_ss.head()

In [ ]:
pca_df_3.loc[pca_df_3["subreddit"]=="COVID19",]

In [ ]:
def transform_coords(x_dat, y_dat, trans_coef=200):
    #thx to https://matplotlib.org/3.2.1/api/transformations.html#matplotlib.transforms.Transform
    # https://matplotlib.org/3.1.1/tutorials/advanced/transforms_tutorial.html
    fig, ax = plt.subplots()
    out = ax.transData.transform((x_dat[0],y_dat[0]))
    for i in range(0,len(x_dat)):
        t = ax.transData.transform((x_dat[i],y_dat[i]))
        out = np.vstack((out,t))
    out = out/trans_coef
    plt.close(fig)
    return(out)

In [ ]:
#transform_coords(x_test,y_test, trans_coef=35)

In [ ]:
def simple_axis_interpol(l):
    out = []
    min_x = min(l)
    max_x = max(l)
    for x in l:
        x_shifted = x - min_x
        out.append(x_shifted / max_x)
    return(out)

In [ ]:
#simple_axis_interpol(use_flow_ss["PCA_x"].tolist())

In [ ]:
def covid_sankey_position(flow_dat, covid_x, covid_y, trans_coef = 35, 
                 transform = True,interpol=False, x_shift = 0.01, y_shift=0.01):
    # flow_dat: some datframe where you have subreddits and acording flows ("Freq" in this case)
    # covid_x/covid_y: the x dimension of PCA reduction that will be used as coordinate
    # trans_coef: a coefficient that "tweaks" the transformation from matplotlib. the transform is in the 
    # thrid decimal area, so I need to reduce it
    # interpole: should interpolation be used (thx to raphi)
    # x_shift/ y_shift: other "tweaking" variables
    #RETURNS: Sankey
    label_list = flow_dat["subreddit_y"].tolist()
    label_list.insert(0,"COVID19")
    x_cor_list = flow_dat["PCA_x"] + x_shift# adding values doesnt help
    x_cor_list =  x_cor_list.tolist()
    y_cor_list =  flow_dat["PCA_y"] + y_shift
    y_cor_list = y_cor_list.tolist()
    x_cor_list.insert(0,covid_x + x_shift)
    y_cor_list.insert(0,covid_y + y_shift)
    #transform
    if transform == True:
        if interpol:
            x_cor_list = simple_axis_interpol(x_cor_list)
            y_cor_list = simple_axis_interpol(y_cor_list)
        else:
            t = transform_coords(x_cor_list, y_cor_list, trans_coef=trans_coef)
            x_cor_list = t[:,0].tolist()
            y_cor_list = t[:,1].tolist()
    #
    fig = go.Figure(data=[go.Sankey(
        arrangement = "snap",
        node = {
          "thickness": 20,
         "line": dict(color = "black", width = 0.5),
          "label": label_list,
            "x": x_cor_list,
            "y":y_cor_list,
            "pad": 0.5
        },
        link = {
          "source":list(range(1,len(label_list))), # indices correspond to labels, eg A1, A2, A2, B1, ...
          "target": [0] * len(label_list),
          "value" : flow_dat["Freq"].tolist()
            })])
    #fig.update_layout(title_text="Basic Sankey Diagram", font_size=10)
    fig.show()
    #return(x_cor_list, y_cor_list)

In [ ]:
#TRANSFORMATION METHOD 1
x = pca_df_3["PCA_x"].loc[pca_df_3["subreddit"]=="COVID19",].tolist()[0]
y = pca_df_3["PCA_y"].loc[pca_df_3["subreddit"]=="COVID19",].tolist()[0]
#covid_sankey_position(use_flow_ss.loc[use_flow_ss["Freq"]>200,], x, y, trans_coef=40)
covid_sankey_position(use_flow_ss, x, y, trans_coef=30, transform= True, interpol = False, 
                      x_shift = -0.02, y_shift= -0.07)

In [ ]:
covid_sankey_position(use_flow_ss.loc[use_flow_ss["Freq"]>150,], x, y, trans_coef=30, transform= True, interpol = False, 
                      x_shift = -0.02, y_shift= -5)

In [ ]:
#TRANSFORMATION METHOD 2(INTERPOLATION)
covid_sankey_position(use_flow_ss, x, y, trans_coef=30, transform= True, interpol=True,
                      x_shift = +0.15, y_shift=-0.4)

In [ ]:
covid_sankey_position(use_flow_ss.loc[use_flow_ss["Freq"]>150,], x, y, trans_coef=30, transform= True, interpol=True,
                      x_shift = +0.2, y_shift=-0.4)